# Natural Language Inference

## Running an ESIM-lite BiLSTM Attention Model on Unseen Data

This notebook runs the trained Natural Language Inference (NLI) model on unseen test data.
The tokenizer and configuration saved during training are loaded to ensure the same preprocessing steps are applied.
The trained model then generates predictions which are saved to a CSV file for submission.

## Path

This section defines the file paths for the test dataset and the output prediction file.
These paths are used to load the unseen data and save the generated predictions.

In [15]:
import os

# Base directory (notebook is inside category_B/)
BASE_DIR = ".."

# Paths
DATA_DIR = os.path.join(BASE_DIR, "data")
TEST_CSV  = os.path.join(DATA_DIR, "test.csv")

OUT_DIR  = os.path.join(BASE_DIR, "outputs/category_B")
os.makedirs(OUT_DIR, exist_ok=True)

PRED_OUT = os.path.join(OUT_DIR, "Group_22_B.csv")

print("TEST_CSV:", TEST_CSV)
print("OUT_DIR:", OUT_DIR)
print("PRED_OUT:", PRED_OUT)

# Check required files exist
required_files = ["best_model.keras", "tokenizer.pkl", "config.json"]
missing = [f for f in required_files if not os.path.exists(os.path.join(OUT_DIR, f))]

if missing:
    raise FileNotFoundError(
        f"Missing required files: {missing}. "
        f"Please download the model artifacts from the link in the README "
        f"and place them in: {OUT_DIR}"
    )

TEST_CSV: ./data/test.csv
OUT_DIR: ./outputs/category_B
PRED_OUT: ./outputs/category_B/Group_22_B.csv


## Imports

The required libraries are imported for loading the trained model, processing the text data, and padding token sequences.

In [16]:
# Standard libraries for data handling and model loading
import json
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf

# Used to pad token sequences to fixed length
from tensorflow.keras.preprocessing.sequence import pad_sequences

## Load Configuration

The configuration file saved during training is loaded so that the same sequence length and preprocessing settings are used during prediction.

In [17]:
# Load training configuration saved during training
cfg_path = os.path.join(OUT_DIR, "config.json")
with open(cfg_path, "r") as f:
    cfg = json.load(f)

# Use the same sequence length as used during training
MAX_LEN = int(cfg["MAX_LEN"])

print("Loaded config:", cfg_path)
print("MAX_LEN:", MAX_LEN)

Loaded config: ./outputs/category_B/config.json
MAX_LEN: 35


## Load Tokenizer

The tokenizer created during training is loaded to ensure the same vocabulary mapping is used when converting text into sequences.

In [18]:
# Load the tokenizer used during training
tok_path = os.path.join(OUT_DIR, "tokenizer.pkl")

with open(tok_path, "rb") as f:
    tokenizer = pickle.load(f)

print("Loaded tokenizer:", tok_path)

Loaded tokenizer: ./outputs/category_B/tokenizer.pkl


## Tokenisation Function

This function converts text into sequences of token IDs using the trained tokenizer and pads them to the fixed maximum sequence length.

In [19]:
# Convert text to token sequences and pad to MAX_LEN
def tokenize_and_pad(texts, tokenizer, max_length):

    # Convert words to integer token IDs
    seqs = tokenizer.texts_to_sequences(texts.astype(str).tolist())

    # Pad sequences so they all have the same length
    return pad_sequences(seqs, maxlen=max_length, padding="post", truncating="post")

## Load Model

The trained model saved during training is loaded so it can be used to generate predictions.

In [20]:
# Path to the best model checkpoint saved during training
model_path = os.path.join(OUT_DIR, "best_model.keras")

# Load the trained model
model = tf.keras.models.load_model(model_path, compile=False)

print("Loaded model:", model_path)

Loaded model: ./outputs/category_B/best_model.keras


## Generate Predictions

The trained model is applied to the unseen test dataset.  
Predicted labels are generated and saved to a CSV file in the required submission format.

In [21]:
# Function to save predictions in submission format
def save_predictions_only(predictions, output_path):
    """
    Save predictions to a CSV file with a single 'prediction' column.
    """

    df = pd.DataFrame({"prediction": predictions.flatten()})
    df.to_csv(output_path, index=False)

    print(f"Predictions saved to {output_path}")

In [22]:
# Load test dataset (no labels)
test_df = pd.read_csv(TEST_CSV)

# Convert text into padded sequences
X_test_p = tokenize_and_pad(test_df["premise"], tokenizer, MAX_LEN)
X_test_h = tokenize_and_pad(test_df["hypothesis"], tokenizer, MAX_LEN)

# Predict probabilities
probs = model.predict([X_test_p, X_test_h], batch_size=256, verbose=0).ravel()

# Convert probabilities to binary predictions using threshold 0.5
pred = (probs >= 0.5).astype(int)

# Save predictions
save_predictions_only(pred, PRED_OUT)

# Print total predictions generated
print("Total predictions:", len(pred))

Predictions saved to ./outputs/category_B/Group_22_B.csv
Total predictions: 3302
